# Journal Finder - Data Mining Project

Bu notebook, verilen abstract metnine gore en ilgili 5 dergiyi oneren bir `TF-IDF + Cosine Similarity` yaklasimini uygular.

## Notebook Akisi
1. Veriyi yukleme
2. Hizli veri kontrolu
3. Oneri modeli kurma
4. Ornek abstract ile top-5 dergi onerisi
5. Basit `Hit@5` degerlendirmesi

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
from sklearn.model_selection import train_test_split

DATA_PATH = "../outputs/base_articles_clean.csv"

df = pd.read_csv(DATA_PATH)
print("Rows:", len(df))
print("Columns:", list(df.columns))
df.head(3)

In [ ]:
# Kisa veri kontrolu
print(df[["journal_name", "pub_year"]].describe(include="all"))
print("\nTop 10 journals by count:")
print(df["journal_name"].value_counts().head(10))

In [ ]:
class JournalRecommender:
    def __init__(self, max_features=50000, ngram_range=(1, 2), min_df=3, max_df=0.9):
        self.vectorizer = TfidfVectorizer(
            max_features=max_features,
            ngram_range=ngram_range,
            min_df=min_df,
            max_df=max_df,
            norm="l2",
        )
        self.doc_matrix = None
        self.journals = None

    def fit(self, data, text_col="abstract_clean", journal_col="journal_name"):
        work = data[[text_col, journal_col]].dropna().copy()
        work[text_col] = work[text_col].astype(str).str.strip()
        work[journal_col] = work[journal_col].astype(str).str.strip()
        work = work[(work[text_col] != "") & (work[journal_col] != "")].copy()

        self.doc_matrix = self.vectorizer.fit_transform(work[text_col])
        self.journals = work[journal_col].reset_index(drop=True)
        return self

    def recommend(self, query_abstract, top_k=5, candidate_pool=300):
        query_vec = self.vectorizer.transform([query_abstract])
        sims = linear_kernel(query_vec, self.doc_matrix).flatten()

        candidate_pool = min(candidate_pool, len(sims))
        top_doc_idx = sims.argsort()[::-1][:candidate_pool]

        score_by_journal = {}
        count_by_journal = {}
        for idx in top_doc_idx:
            journal = self.journals.iloc[idx]
            score = float(sims[idx])
            score_by_journal[journal] = score_by_journal.get(journal, 0.0) + score
            count_by_journal[journal] = count_by_journal.get(journal, 0) + 1

        ranked = sorted(score_by_journal.items(), key=lambda x: x[1], reverse=True)[:top_k]
        return pd.DataFrame(
            [
                {
                    "rank": i + 1,
                    "journal_name": j,
                    "score_sum": s,
                    "matched_docs": count_by_journal[j],
                }
                for i, (j, s) in enumerate(ranked)
            ]
        )

recommender = JournalRecommender().fit(df)
print("Model ready")

In [ ]:
# Ornek sorgu
query = """
This study proposes a machine learning based intrusion detection framework
for cloud computing environments using feature selection and ensemble classification.
"""

recommender.recommend(query, top_k=5)

In [ ]:
# Basit Hit@5 degerlendirmesi
# Not: Bu hizli bir benchmark; daha iyi sonuc icin tuning yapilabilir.

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
model_eval = JournalRecommender().fit(train_df)

sample_test = test_df.sample(n=200, random_state=42)

hits = 0
for _, row in sample_test.iterrows():
    pred = model_eval.recommend(row["abstract_clean"], top_k=5)
    pred_journals = set(pred["journal_name"].tolist())
    if row["journal_name"] in pred_journals:
        hits += 1

hit_at_5 = hits / len(sample_test)
print(f"Hit@5 (sample=200): {hit_at_5:.4f}")